In [ ]:
import pickle

## Clustering NP Atlas with MAP4

In [ ]:
with open(f"{fname}.html", encoding='utf-8') as fh:
    html = fh.read()

replacements = [
    (
        """          this.el.search.addEventListener('click', e => {
            let searchTerm = prompt('Please enter a search term:', '');
            const results = this.search(searchTerm);

            for (const [name, indices] of Object.entries(results)) {
              if (!name in this.ohIndexMap) return;
              indices.forEach(index => {
                this.octreeHelpers[this.ohIndexMap[name]].addSelected(index);
              });
            }
            
            e.preventDefault();
            return false;
          });""",
        """          this.el.search.addEventListener('click', e => {
            const searchTerm = prompt('Please enter an NPA ID:', '');
            this.searchAndFocus(searchTerm);
            
            e.preventDefault();
            return false;
          });"""
    ),
    (
        """          window.addEventListener('keydown', e => {
            if ((e.keyCode == 114) || (e.ctrlKey && e.keyCode == 70)) {
              this.search();
              e.preventDefault();
              return false;
            }
          });""",
        """          window.addEventListener('keydown', e => {
            if ((e.keyCode == 114) || (e.ctrlKey && e.keyCode == 70)) {
              const searchTerm = prompt('Please enter an NPA ID:', '');
              this.searchAndFocus(searchTerm);
              e.preventDefault();
              return false;
            }
          });"""
    ),
    (
        """        search(term) {
          let results = {}
          let re = new RegExp(term, 'i');

          Object.keys(data).forEach(name => {
            if (!('labels' in data[name])) return;
            results[name] = []
            data[name]['labels'].forEach((label, i) => {
              if (re.test(label))
                results[name].push(i)
            });
          });

          return results;
        }""",
        """        clearSelection() {
          let items = [...this.selectedItems];
          items.forEach(item => {
            this.octreeHelpers[item.source].removeSelected(item.item.index);
          });
        }

        searchAndFocus(term) {
          let results = this.search(term);
          let firstMatch = null;

          this.clearSelection();

          for (const [name, indices] of Object.entries(results)) {
            if (!(name in this.ohIndexMap)) continue;

            indices.forEach(index => {
              if (firstMatch === null) {
                firstMatch = { name, index };
              }

              this.octreeHelpers[this.ohIndexMap[name]].addSelected(index);
            });
          }

          if (firstMatch === null) {
            if ((term || '').trim()) {
              alert('No molecules matched that NPA ID.');
            }
            return;
          }

          let selectedIndex = this.getSelectedIndex(
            this.ohIndexMap[firstMatch.name],
            firstMatch.index
          );

          if (selectedIndex !== null) {
            this.updateSelected(selectedIndex);
          }
        }

        search(term) {
          let results = {};
          let normalizedTerm = (term || '').trim().toLowerCase();

          Object.keys(data).forEach(name => {
            if (!('info' in data[name])) return;
            results[name] = [];
            if (!normalizedTerm) return;

            data[name]['info'].forEach((label, i) => {
              if ((label || '').toLowerCase().includes(normalizedTerm)) {
                results[name].push(i);
              }
            });
          });

          return results;
        }"""
    ),
    (
        """        setSelectedContent(fullLabel, labelIndex, selectedLabels, img) {
          this.el.selectedContainer.innerHTML = '';
          if (img)
            this.el.selectedContainer.appendChild(img);

          fullLabel.forEach((l, i) => {
            if (i === labelIndex) return;
            if (selectedLabels && selectedLabels[i]) {
              this.el.selectedContainer.appendChild(
                Faerun.createElement('div', { classes: 'label', content: selectedLabels[i] })
              );
            }
            this.el.selectedContainer.appendChild(
              Faerun.createElement('div', { classes: 'content', content: l })
            );
          });

          // Update the indicator
          this.updateSelectedIndicators();
        }""",
        """        setSelectedContent(fullInfo, selectedLabels, img) {
          this.el.selectedContainer.innerHTML = '';
          if (img)
            this.el.selectedContainer.appendChild(img);

          fullInfo.forEach((l, i) => {
            if (selectedLabels && selectedLabels[i]) {
              this.el.selectedContainer.appendChild(
                Faerun.createElement('div', { classes: 'label', content: selectedLabels[i] })
              );
            }
            this.el.selectedContainer.appendChild(
              Faerun.createElement('div', { classes: 'content', content: l })
            );
          });

          // Update the indicator
          this.updateSelectedIndicators();
        }"""
    ),
    (
        """          let fullLabel = data[phName].labels[item.item.index].split('__');

          let labelIndex = meta.label_index[seriesState];
          let titleIndex = meta.title_index[seriesState];
          let selectedLabels = meta.selected_labels[seriesState];

          SmilesDrawer.parse(fullLabel[labelIndex], tree => {
            smilesDrawer.draw(tree, 'smiles-canvas', 'dark', false);
            let img = Faerun.createElement('img');
            img.src = this.el.smilesCanvas.toDataURL();
            
            this.el.selectedCurrent.innerHTML = current + 1;
            this.el.selectedTotal.innerHTML = n;
            this.el.selectedTitle.innerHTML = fullLabel[titleIndex];

            this.setSelectedContent(fullLabel, labelIndex, selectedLabels, img);
          }, err => { console.log(err); });""",
        """          let fullLabel = data[phName].labels[item.item.index].split('__');
          let fullInfo = data[phName].info[item.item.index].split('__');

          let labelIndex = meta.label_index[seriesState];
          let selectedLabels = meta.selected_labels[seriesState];

          SmilesDrawer.parse(fullLabel[labelIndex], tree => {
            smilesDrawer.draw(tree, 'smiles-canvas', 'dark', false);
            let img = Faerun.createElement('img');
            img.src = this.el.smilesCanvas.toDataURL();
            
            this.el.selectedCurrent.innerHTML = current + 1;
            this.el.selectedTotal.innerHTML = n;
            this.el.selectedTitle.innerHTML = fullInfo[0] || '';

            this.setSelectedContent(fullInfo, selectedLabels, img);
          }, err => { console.log(err); });"""
    ),
    (
        """      #selected #selected-container {
        padding: 10px;
        font-size: 0.8em;
      }

      #selected #selected-container img {
        margin-left: -10px;
      }""",
        """      #selected #selected-container {
        padding: 10px;
        font-size: 0.8em;
        user-select: text;
        -webkit-user-select: text;
      }

      #selected #selected-container img {
        margin-left: -10px;
      }"""
    ),
    (
        """      .selected-indicator {
        position: absolute;
        z-index: 999;
        pointer-events: none;
      }

      .selected-indicator .crosshair-x {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        height: 2px;
        width: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }

      .selected-indicator .crosshair-y {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        width: 2px;
        height: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }""",
        """      .selected-indicator {
        position: absolute;
        z-index: 999;
        pointer-events: none;
      }

      .selected-indicator .selected-ring {
        position: absolute;
        top: -6px;
        left: -6px;
        width: 200%;
        height: 200%;
        border: 2px solid #fff;
        border-radius: 50%;
        box-sizing: border-box;
      }

      .selected-indicator .crosshair-x {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        height: 2px;
        width: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }

      .selected-indicator .crosshair-y {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        width: 2px;
        height: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }"""
    ),
    (
        """          let indicatorElement = Faerun.createElement(
            'div', 
            { classes: 'selected-indicator' },
            [
              Faerun.createElement('div', { classes: 'crosshair-x' }),
              Faerun.createElement('div', { classes: 'crosshair-y' })
            ]
          );""",
        """          let indicatorElement = Faerun.createElement(
            'div', 
            { classes: 'selected-indicator' },
            [
              Faerun.createElement('div', { classes: 'selected-ring' }),
              Faerun.createElement('div', { classes: 'crosshair-x' }),
              Faerun.createElement('div', { classes: 'crosshair-y' })
            ]
          );"""
    )
]

for old, new in replacements:
    if old not in html:
        raise ValueError('Expected HTML snippet not found while patching html file')
    html = html.replace(old, new, 1)

with open(f"{fname}.html", 'w', encoding='utf-8') as fh:
    fh.write(html)

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import umap

def export_js(df):
    X = np.stack(df["FP_map4"].map(np.asarray).values).astype(np.float32)

    reducer = umap.UMAP(
    n_components=3,
    n_neighbors=80,
    min_dist=0.1,
    metric="manhattan",
    random_state=7,
    n_jobs=1,)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module="umap\\.umap_")
        xyz = reducer.fit_transform(X)

    xyz_centered = xyz - xyz.mean(axis=0, keepdims=True)

    x = xyz_centered[:, 0]
    y = xyz_centered[:, 1]
    z = xyz_centered[:, 2]

    r = np.sqrt(x * x + y * y + z * z)

    eps = 1e-12
    theta = np.arccos(np.clip(z / np.maximum(r, eps), -1.0, 1.0))
    phi = np.arctan2(y, x)

    sphere_radius = 3.2

    r_norm = (r - r.min()) / (r.max() - r.min() + 1e-12)
    r_web = sphere_radius * np.cbrt(r_norm)

    x_web = r_web * np.sin(theta) * np.cos(phi)
    y_web = r_web * np.sin(theta) * np.sin(phi)
    z_web = r_web * np.cos(theta)

    cluster_labels = df["cluster_id"].fillna(-1).astype(int).values

    points = []
    for i in range(len(df)):
        points.append({
        "id": df.iloc[i]["NPA_ID"],
        "smiles": df.iloc[i]["ISOSMILES"],
        "x": round(float(x_web[i]), 4),
        "y": round(float(y_web[i]), 4),
        "z": round(float(z_web[i]), 4),
        "cluster_label": int(cluster_labels[i]),
    })

    points_js = "window.CHEMSPACE_DATA = " + json.dumps(points, ensure_ascii=False) + ";"

    Path('chemspace_data.js').write_text(points_js, encoding="utf-8")

In [ ]:
df = df_frags.loc[df_frags["FP_map4"].notna()].copy().reset_index(drop=True)
export_js(df)